In [ ]:
# ============================================================
# COLAB SETUP
# This cell only prepares Colab, the project repo, and ConLID.
# It does NOT change the original end-to-end fine-tuning method.
# ============================================================

import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

PROJECT_REPO_URL = (
    "https://github.com/Maleesha-K/"
    "Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit.git"
)

CONLID_REPO_URL = "https://github.com/epfl-nlp/language-identification.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    # Clone the project into Colab local storage.
    # This is faster than running the whole repo from Google Drive.
    PROJECT_ROOT = Path("/content/ssli_repo")

    if not (PROJECT_ROOT / ".git").exists():
        if PROJECT_ROOT.exists():
            import shutil
            shutil.rmtree(PROJECT_ROOT)

        print("Cloning project repository...")
        subprocess.run(
            [
                "git", "clone", "--depth", "1",
                PROJECT_REPO_URL,
                str(PROJECT_ROOT),
            ],
            check=True,
        )
    else:
        print("Project repo already exists. Pulling latest main...")
        subprocess.run(
            ["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only", "origin", "main"],
            check=True,
        )

    DATA_PIPELINE_ROOT = PROJECT_ROOT / "data_pipeline"
    os.chdir(DATA_PIPELINE_ROOT)

    # Install packages required by this notebook.
    print("Installing notebook dependencies...")
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "pandas",
            "scikit-learn",
            "evaluate",
            "huggingface_hub",
            "safetensors",
            "tqdm",
        ],
        check=True,
    )

    # Clone official ConLID implementation.
    REPO_DIR_PATH = DATA_PIPELINE_ROOT / "models" / "benchmark" / "ConLID" / "repo"
    REPO_DIR_PATH.parent.mkdir(parents=True, exist_ok=True)

    if not (REPO_DIR_PATH / ".git").exists():
        if REPO_DIR_PATH.exists():
            import shutil
            shutil.rmtree(REPO_DIR_PATH)

        print("Cloning official ConLID repository...")
        subprocess.run(
            [
                "git", "clone", "--depth", "1",
                CONLID_REPO_URL,
                str(REPO_DIR_PATH),
            ],
            check=True,
        )
    else:
        print("ConLID repository already exists.")

    # NOTE: ConLID's requirements.txt is deliberately NOT installed.
    #
    # It pins fasttext-wheel==0.9.2, which has no wheel for the Python
    # version Colab now ships and fails to build from source. It also
    # hard-pins transformers/huggingface_hub/datasets to older releases,
    # which would downgrade Colab's working install.
    #
    # models/benchmark/ConLID/repo/model.py imports only:
    #     json, os, re, numpy, torch, torch.nn, safetensors.torch
    # all of which are already present. Nothing else in that file is
    # needed to load the checkpoint or fine-tune it, so we verify the
    # real imports instead of installing the pinned list.
    print("Verifying ConLID model dependencies...")

    missing = []
    for module in ["torch", "numpy", "safetensors"]:
        try:
            __import__(module)
        except ImportError:
            missing.append(module)

    if missing:
        print(f"Installing missing dependencies: {missing}")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *missing],
            check=True,
        )
    else:
        print("  torch, numpy, safetensors already available.")

    print("\nColab setup complete.")
    print("Project root:", PROJECT_ROOT)
    print("data_pipeline:", DATA_PIPELINE_ROOT)

else:
    # Local fallback: find data_pipeline from the current directory.
    cwd = Path.cwd().resolve()

    DATA_PIPELINE_ROOT = None
    for p in [cwd, *cwd.parents]:
        if p.name == "data_pipeline":
            DATA_PIPELINE_ROOT = p
            break

        candidate = p / "data_pipeline"
        if candidate.exists():
            DATA_PIPELINE_ROOT = candidate.resolve()
            break

    if DATA_PIPELINE_ROOT is None:
        raise FileNotFoundError("Could not locate the data_pipeline folder.")

    PROJECT_ROOT = DATA_PIPELINE_ROOT.parent
    REPO_DIR_PATH = DATA_PIPELINE_ROOT / "models" / "benchmark" / "ConLID" / "repo"
    os.chdir(DATA_PIPELINE_ROOT)

print("Current working directory:", os.getcwd())


In [ ]:
# ============================================================
# CONFIGURATION
# Original fine-tuning hyperparameters are unchanged.
# ============================================================

from pathlib import Path

batch_size = 8
accumulation_steps = 4
learning_rate = 1e-4
num_epochs = 10

# Your known Google Drive data location.
# The notebook also searches under /content/drive/MyDrive/SSLI if needed.
DRIVE_DATA_ROOT = Path("/content/drive/MyDrive/SSLI")
PREFERRED_DATA_DIR = DRIVE_DATA_ROOT / "processed" / "public_shared"


def find_data_file(names):
    """
    Find a required dataset file without silently creating a new split.
    Preference:
      1. /content/drive/MyDrive/SSLI/processed/public_shared/
      2. anywhere under /content/drive/MyDrive/SSLI/
      3. current data_pipeline folder (local fallback)
    """
    if isinstance(names, str):
        names = [names]

    # Preferred known location first.
    for name in names:
        candidate = PREFERRED_DATA_DIR / name
        if candidate.exists():
            return candidate

    # Search Drive.
    if DRIVE_DATA_ROOT.exists():
        for name in names:
            matches = sorted(DRIVE_DATA_ROOT.rglob(name))
            if matches:
                return matches[0]

    # Local fallback.
    for name in names:
        for candidate in [
            DATA_PIPELINE_ROOT / "datasets" / "finetuning" / name,
            DATA_PIPELINE_ROOT / "test_dataset_folder" / name,
        ]:
            if candidate.exists():
                return candidate

    return None


TRAIN_PATH = find_data_file("train.csv")

# Validation is NOT used for training gradients.
# We only locate an existing validation file; no new split is created.
VAL_PATH = find_data_file(
    ["val.csv", "validation.csv", "val_mixed.jsonl"]
)

if TRAIN_PATH is None:
    raise FileNotFoundError(
        "train.csv was not found. Expected it under "
        "/content/drive/MyDrive/SSLI/processed/public_shared/ "
        "or elsewhere under /content/drive/MyDrive/SSLI/."
    )

if VAL_PATH is None:
    raise FileNotFoundError(
        "No existing validation file was found. "
        "Expected val.csv, validation.csv, or val_mixed.jsonl "
        "under /content/drive/MyDrive/SSLI/."
    )

# Save the trained model permanently in Google Drive on Colab.
if IN_COLAB:
    output_model_dir = Path(
        "/content/drive/MyDrive/SSLI/models/finetuned/conlid_end_to_end_script_fixed"
    )
else:
    output_model_dir = DATA_PIPELINE_ROOT / "models" / "finetuned" / "conlid"

output_model_dir.mkdir(parents=True, exist_ok=True)

print("TRAIN_PATH:", TRAIN_PATH)
print("VAL_PATH:  ", VAL_PATH)
print("OUTPUT:    ", output_model_dir)
print()
print("Hyperparameters:")
print("batch_size =", batch_size)
print("accumulation_steps =", accumulation_steps)
print("learning_rate =", learning_rate)
print("num_epochs =", num_epochs)


TRAIN_PATH: /content/drive/MyDrive/SSLI/processed/public_shared/train.csv
VAL_PATH:   /content/drive/MyDrive/SSLI/processed/public_shared/val.csv
OUTPUT:     /content/drive/MyDrive/SSLI/models/finetuned/conlid_end_to_end_script_fixed

Hyperparameters:
batch_size = 8
accumulation_steps = 4
learning_rate = 0.0001
num_epochs = 10


In [ ]:
import os
import sys
import json
import torch
import torch.nn as nn
import pandas as pd
from huggingface_hub import snapshot_download

REPO_DIR = str(REPO_DIR_PATH)

if not os.path.exists(REPO_DIR):
    raise RuntimeError(
        f"{REPO_DIR} not found. Re-run the Colab setup cell."
    )

if REPO_DIR not in sys.path:
    sys.path.append(REPO_DIR)

from model import ConLID  # noqa: E402

print("Downloading/checking ConLID checkpoint...")

checkpoint_dir = os.path.join(
    REPO_DIR,
    "checkpoints",
    "conlid"
)

snapshot_download(
    repo_id="epfl-nlp/ConLID",
    local_dir=checkpoint_dir
)

print("Loading ConLID model...")

model = ConLID.from_pretrained(
    dir=checkpoint_dir
)

# Explicitly use the Colab GPU when available.
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# ORIGINAL TARGET MAPPING FROM THE UPLOADED NOTEBOOK
# Kept unchanged to preserve the original fine-tuning method.
# ------------------------------------------------------------
TARGET_LANGUAGES = {
    "eng": "eng_Latn",
    "hin": "hin_Deva",
    "arb": "arb_Arab",
    "fra": "fra_Latn",
    "deu": "deu_Latn",
    "ben": "ben_Beng",
    "tam": "tam_Taml",

    # Sinhala-script target languages
    "sin": "sin_Sinh",
    "pli": "pli_Sinh",
    "san": "san_Sinh",
}

# Your CSV uses readable labels.
# This only converts the input spelling to the original notebook's
# short codes before TARGET_LANGUAGES is applied.
DATASET_LABEL_ALIASES = {
    "sinhala": "sin",
    "pali": "pli",
    "sanskrit": "san",
    "sin": "sin",
    "pli": "pli",
    "san": "san",
}

# ------------------------------------------------------------
# ORIGINAL CLASSIFICATION-HEAD EXPANSION LOGIC
# ------------------------------------------------------------
existing_labels = list(model.id2label.values())
added_labels = []

for short_code, long_code in TARGET_LANGUAGES.items():
    if long_code not in existing_labels:
        idx = len(model.id2label)
        model.id2label[idx] = long_code
        existing_labels.append(long_code)
        added_labels.append(long_code)

if added_labels:
    import numpy as np

    print(
        f"Expanding classification head to add "
        f"{len(added_labels)} new labels: {added_labels}"
    )

    old_out_features = model.fc.out_features
    new_out_features = len(model.id2label)

    new_fc = nn.Linear(
        model.fc.in_features,
        new_out_features
    )

    new_fc.weight.data[:old_out_features] = model.fc.weight.data
    new_fc.bias.data[:old_out_features] = model.fc.bias.data

    nn.init.xavier_uniform_(
        new_fc.weight.data[old_out_features:]
    )
    nn.init.zeros_(
        new_fc.bias.data[old_out_features:]
    )

    model.fc = new_fc.to(device)
    model.convert_id2label = np.vectorize(
        model.id2label.get
    )

print(f"Model ready. Total labels: {len(model.id2label)}")

print("\nChecking important labels:")

for label in [
    "sin_Sinh",
    "pli_Sinh",
    "san_Sinh",
    "pli_Latn",
    "san_Deva",
]:
    print(
        f"{label}:",
        label in model.id2label.values()
    )

Downloading/checking ConLID checkpoint...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

labels.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/161 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.15G [00:00<?, ?B/s]

Loading ConLID model...
Device: cuda
GPU: Tesla T4
Expanding classification head to add 2 new labels: ['pli_Sinh', 'san_Sinh']
Model ready. Total labels: 2101

Checking important labels:
sin_Sinh: True
pli_Sinh: True
san_Sinh: True
pli_Latn: False
san_Deva: True


In [ ]:
from torch.utils.data import Dataset, DataLoader

# Reverse lookup for label string to ID
label2id = {
    v: k
    for k, v in model.id2label.items()
}

MAX_TEXT_CHARS = 2000


class ConLIDDataset(Dataset):
    """
    Same tokenization / n-gram preparation as the original notebook.

    Only the file reader was made Colab-safe:
    - train.csv is supported directly
    - an existing CSV or JSONL validation file is supported
    """

    def __init__(self, file_path, model, label_map):
        self.records = []
        self.model = model

        file_path = Path(file_path)

        if not file_path.exists():
            raise FileNotFoundError(
                f"Dataset file not found: {file_path}"
            )

        # Read the provided file without changing the split.
        if file_path.suffix.lower() == ".csv":
            df = pd.read_csv(file_path)

            required_columns = {"text", "label"}
            missing = required_columns - set(df.columns)

            if missing:
                raise ValueError(
                    f"{file_path} is missing columns: {missing}"
                )

            rows = df[["text", "label"]].to_dict("records")

        elif file_path.suffix.lower() == ".jsonl":
            rows = []

            with open(
                file_path,
                "r",
                encoding="utf-8"
            ) as f:
                for line in f:
                    line = line.strip()

                    if line:
                        rows.append(
                            json.loads(line)
                        )
        else:
            raise ValueError(
                f"Unsupported dataset format: {file_path.suffix}"
            )

        skipped_counts = {}

        for rec in rows:
            text = str(rec["text"])
            raw_label = str(rec["label"]).strip()

            # Convert full CSV names such as "pali" -> "pli".
            alias_key = raw_label.lower()
            short_label = DATASET_LABEL_ALIASES.get(
                alias_key,
                raw_label
            )

            # Map project labels to their Sinhala-script ConLID classes:
            # pli -> pli_Sinh
            # san -> san_Sinh
            # sin -> sin_Sinh
            mapped_label = label_map.get(
                short_label,
                short_label
            )

            if mapped_label in label2id:
                self.records.append(
                    (
                        text,
                        label2id[mapped_label]
                    )
                )
            else:
                skipped_counts[mapped_label] = (
                    skipped_counts.get(mapped_label, 0) + 1
                )

        print(f"\nLoaded: {file_path}")
        print(f"Input rows: {len(rows)}")
        print(f"Usable rows: {len(self.records)}")

        if skipped_counts:
            print("Skipped labels:", skipped_counts)

        if len(self.records) == 0:
            raise ValueError(
                f"No usable records were loaded from {file_path}. "
                "Check the label values."
            )

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        text, label = self.records[idx]

        text = text[:MAX_TEXT_CHARS]

        tokens = self.model._tokenize(text)
        ids = self.model._tokens2ngrams(tokens)

        return (
            torch.tensor(
                ids,
                dtype=torch.int
            ),
            torch.tensor(
                label,
                dtype=torch.long
            ),
        )


def collate_fn(batch):
    ids, labels = zip(*batch)

    max_len = max(
        len(x)
        for x in ids
    )

    padded_ids = [
        torch.cat(
            [
                x,
                torch.full(
                    (max_len - len(x),),
                    model.pad_id,
                    dtype=torch.int
                ),
            ]
        )
        for x in ids
    ]

    return (
        torch.stack(padded_ids),
        torch.stack(labels)
    )


print("\nLoading Datasets...")

train_dataset = ConLIDDataset(
    TRAIN_PATH,
    model,
    TARGET_LANGUAGES
)

val_dataset = ConLIDDataset(
    VAL_PATH,
    model,
    TARGET_LANGUAGES
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn
)

print()
print(f"Train size: {len(train_dataset)}")
print(f"Validation size: {len(val_dataset)}")



Loading Datasets...

Loaded: /content/drive/MyDrive/SSLI/processed/public_shared/train.csv
Input rows: 60285
Usable rows: 60285

Loaded: /content/drive/MyDrive/SSLI/processed/public_shared/val.csv
Input rows: 6986
Usable rows: 6986

Train size: 60285
Validation size: 6986


In [ ]:
# ============================================================
# MODEL SAVING + VERIFICATION
#
# The previous run trained successfully but never wrote the model
# to disk, leaving an empty output folder. These helpers save the
# model and verify the bytes actually landed.
# ============================================================
import json
import os
from pathlib import Path

import safetensors.torch


def save_model_checkpoint(model, out_dir, tag=""):
    """Save model + configs, then verify the files exist and are non-empty.

    Raises RuntimeError if anything failed to write, so a silent save
    failure can never again look like a successful run.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    weights_path = out_dir / "model.safetensors"
    safetensors.torch.save_model(model, str(weights_path))

    # IMPORTANT - vocab_size must describe the EMBEDDING TABLE, not len(vocab).
    #
    # ConLID.__init__ builds the embedding with (vocab_size + bucket) rows,
    # but from_pretrained() then overwrites model.vocab_size with
    # len(model.vocab) AFTER loading. So the live attribute is the smaller
    # word-count, and writing it into config.json produces a config that
    # rebuilds a too-small embedding and fails to reload with:
    #     size mismatch for embedding.weight
    #
    # Derive it from the actual tensor instead of trusting the attribute.
    embedding_rows = model.embedding.weight.shape[0]
    true_vocab_size = embedding_rows - model.bucket

    with open(out_dir / "config.json", "w") as f:
        json.dump({
            "vocab_size": true_vocab_size,
            "embedding_size": model.embedding.embedding_dim,
            "num_classes": model.fc.out_features,
            "bucket": model.bucket,
            "min_count": model.min_count,
            "minn": model.minn,
            "maxn": model.maxn,
            "aggr": "mean",
            "pad_id": model.pad_id,
            "unk_id": model.unk_id,
        }, f)

    with open(out_dir / "vocab.json", "w") as f:
        json.dump(model.vocab, f)

    with open(out_dir / "labels.json", "w") as f:
        json.dump({v: k for k, v in model.id2label.items()}, f)

    # --- verify ---
    required = ["model.safetensors", "config.json", "vocab.json", "labels.json"]
    missing = []
    sizes = {}

    for name in required:
        p = out_dir / name
        if not p.exists() or p.stat().st_size == 0:
            missing.append(name)
        else:
            sizes[name] = p.stat().st_size

    if missing:
        raise RuntimeError(
            f"SAVE FAILED - missing or empty: {missing} in {out_dir}. "
            "Do not trust this run; re-check Drive mount and free space."
        )

    label = f" [{tag}]" if tag else ""
    print(f"  Saved{label} to {out_dir}")
    for name in required:
        mb = sizes[name] / (1024 * 1024)
        print(f"    {name:20s} {sizes[name]:>14,d} bytes ({mb:,.1f} MB)")

    return out_dir


def verify_saved_model(out_dir):
    """Reload the saved weights into a fresh ConLID and check the labels."""
    out_dir = Path(out_dir)
    print(f"\nVerifying saved model in {out_dir} ...")

    reloaded = ConLID.from_pretrained(dir=str(out_dir))
    n_labels = len(reloaded.id2label)
    labels = set(reloaded.id2label.values())

    print(f"  Reloaded OK. Total labels: {n_labels}")
    for code in ["sin_Sinh", "pli_Sinh", "san_Sinh", "san_Deva"]:
        print(f"    {code:10s} {code in labels}")

    expected = {"sin_Sinh", "pli_Sinh", "san_Sinh"}
    if not expected.issubset(labels):
        raise RuntimeError(
            f"Saved model is missing target labels: {expected - labels}"
        )

    print("  Verification passed.")
    return reloaded


# Per-epoch checkpoints go here so a Colab disconnect near the end
# of training can never lose the whole run again.
CHECKPOINT_DIR = Path(output_model_dir).parent / (
    Path(output_model_dir).name + "_checkpoints"
)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("Final model dir: ", output_model_dir)
print("Checkpoint dir:  ", CHECKPOINT_DIR)


In [ ]:
import numpy as np
from tqdm.auto import tqdm
import copy
from sklearn.metrics import f1_score

# ============================================================
# ORIGINAL END-TO-END FINE-TUNING
# ALL model parameters remain trainable exactly as before.
#
# Training method, hyperparameters, early stopping and the
# best-model selection are UNCHANGED from the run that produced
# the reported benchmark scores. The only additions are:
#   - a per-epoch checkpoint save (survives a Colab disconnect)
#   - a verified final save (the previous run saved nothing)
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate
)

criterion = nn.CrossEntropyLoss()

best_f1 = -1
best_model_state = None

patience = 2
patience_counter = 0

print("Starting Fine-tuning...")

for epoch in range(num_epochs):

    model.train()
    total_loss = 0

    for step, (input_ids, labels) in enumerate(
        tqdm(
            train_loader,
            desc=f"Epoch {epoch+1}/{num_epochs}"
        )
    ):

        input_ids = input_ids.to(device)
        labels = labels.to(device)

        logits = model(input_ids)

        loss = criterion(
            logits,
            labels
        )

        loss = loss / accumulation_steps

        loss.backward()

        if (
            (step + 1) % accumulation_steps == 0
            or
            (step + 1) == len(train_loader)
        ):
            optimizer.step()
            optimizer.zero_grad()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1} - "
        f"Avg Train Loss: "
        f"{total_loss / len(train_loader):.4f}"
    )

    # Validation
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for input_ids, labels in val_loader:

            input_ids = input_ids.to(device)

            logits = model(input_ids)

            preds = torch.argmax(
                logits,
                dim=-1
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                labels.numpy()
            )

    val_f1 = f1_score(
        all_labels,
        all_preds,
        average="micro",
        zero_division=0
    )

    print(
        f"Epoch {epoch+1} - "
        f"Validation Micro F1: "
        f"{val_f1:.4f}"
    )

    if val_f1 > best_f1:

        best_f1 = val_f1

        best_model_state = copy.deepcopy(
            model.state_dict()
        )

        patience_counter = 0

        # Persist the new best immediately, so a disconnect at any
        # point still leaves the best-so-far model on Drive.
        print(f"  New best F1 ({best_f1:.4f}) - checkpointing...")
        save_model_checkpoint(
            model,
            CHECKPOINT_DIR,
            tag=f"epoch {epoch+1}, F1 {best_f1:.4f}"
        )

    else:

        patience_counter += 1

        if patience_counter >= patience:

            print(
                "Early stopping triggered! "
                f"Restoring best model "
                f"(F1: {best_f1:.4f})"
            )

            model.load_state_dict(
                best_model_state
            )

            break

# ============================================================
# FINAL SAVE  (this is what the previous run was missing)
# ============================================================

# Make sure the weights in memory are the best ones, not the last.
if best_model_state is not None:
    model.load_state_dict(best_model_state)

print("\n" + "=" * 60)
print(f"Training complete. Best validation micro F1: {best_f1:.4f}")
print("=" * 60)

save_model_checkpoint(model, output_model_dir, tag="FINAL")
verify_saved_model(output_model_dir)

print("\nFine-tuning complete. Model saved and verified at:")
print(f"  {output_model_dir}")


## Evaluation: fine-tuned ConLID across ALL benchmark languages

Evaluates the fine-tuned model on every `datasets/preprocessed/*.jsonl` benchmark file, over all target + old languages (not just Sinhala/Pali/Sanskrit), mirroring the "ALL LANGUAGES" reports used for the other finetuned models.

In [ ]:
def find_benchmark_dir():
    candidates = [
        Path("/content/drive/MyDrive/SSLI/processed/public_shared/benchmark_datasets/"),
        DATA_PIPELINE_ROOT / "datasets" / "preprocessed",
    ]

    for candidate in candidates:
        if candidate.exists() and list(candidate.glob("*.jsonl")):
            return candidate

    return None

In [ ]:
from pathlib import Path

BENCHMARK_DIR = Path(
    "/content/drive/MyDrive/SSLI/processed/public_shared/benchmark_datasets/"
)

print("Exists:", BENCHMARK_DIR.exists())

for file in BENCHMARK_DIR.glob("*.jsonl"):
    print(file.name)

Exists: True
commonlid.jsonl
flores_plus.jsonl
wili-2018.jsonl


In [ ]:
# ============================================================
# BENCHMARK EVALUATION
# Evaluates the fine-tuned ConLID model on:
# - CommonLID
# - FLORES+
# - WiLI-2018
# ============================================================

import os
import json
from pathlib import Path

import pandas as pd

from tqdm.auto import tqdm
from IPython.display import display

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
)


# ------------------------------------------------------------
# 1. Exact benchmark paths
# ------------------------------------------------------------

BENCHMARK_DIR = Path(
    "/content/drive/MyDrive/SSLI/processed/public_shared/benchmark_datasets/"
)

BENCHMARK_OUTPUT_DIR = Path(
    "/content/drive/MyDrive/SSLI/benchmark_results/conlid_finetuned_script_fixed/"
)

BENCHMARK_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Verify files
if not BENCHMARK_DIR.exists():
    raise FileNotFoundError(
        f"Benchmark directory not found: {BENCHMARK_DIR}"
    )

benchmark_files = sorted(
    BENCHMARK_DIR.glob("*.jsonl")
)

if not benchmark_files:
    raise FileNotFoundError(
        f"No .jsonl files found inside: {BENCHMARK_DIR}"
    )


print("Benchmark directory:")
print(BENCHMARK_DIR)

print("\nBenchmark files:")
for file in benchmark_files:
    print(" -", file.name)

print("\nResults will be saved to:")
print(BENCHMARK_OUTPUT_DIR)


# ------------------------------------------------------------
# 2. Languages that we want to report
# ------------------------------------------------------------

ALL_BENCHMARK_LANGUAGES = [
    "sinhala",
    "pali",
    "sanskrit",
    "sanskrit_deva",
    "english",
    "tamil",
    "hindi",
    "bengali",
    "arabic",
    "french",
    "german",
]


# ------------------------------------------------------------
# 3. Benchmark label -> readable name
# ------------------------------------------------------------

LABEL_MAPPING_ALL = {
    "sin": "sinhala",
    "pli": "pali",
    "eng": "english",
    "tam": "tamil",
    "hin": "hindi",
    "ben": "bengali",
    "arb": "arabic",
    "fra": "french",
    "deu": "german",
}


# ------------------------------------------------------------
# 4. ConLID model label -> readable name
#
# Keep this consistent with the OLD end-to-end
# fine-tuning notebook.
# ------------------------------------------------------------

MODEL_LABEL_TO_NAME = {
    # Sinhala-script target classes
    "sin_Sinh": "sinhala",
    "pli_Sinh": "pali",
    "san_Sinh": "sanskrit",

    # Original Sanskrit in Devanagari
    "san_Deva": "sanskrit_deva",

    # Original old-language classes
    "eng_Latn": "english",
    "tam_Taml": "tamil",
    "hin_Deva": "hindi",
    "ben_Beng": "bengali",
    "arb_Arab": "arabic",
    "fra_Latn": "french",
    "deu_Latn": "german",
}

# ------------------------------------------------------------
# 5. Map benchmark true labels
# ------------------------------------------------------------

def map_all_label(row):

    lbl = str(
        row.get("label", "")
    ).strip()

    src = str(
        row.get("source", "")
    ).strip()

    # Sanskrit requires special handling.
    #
    # Project Sanskrit sources contain Sanskrit used for
    # our Sinhala-script task.
    #
    # Other Sanskrit benchmark records are treated as
    # Sanskrit-Devanagari.
    if lbl == "san":

        if src in [
            "DCS",
            "SansinNT",
            "SiDiaC-v2",
            "SiDiaC-v2.0",
        ]:
            return "sanskrit"

        return "sanskrit_deva"

    return LABEL_MAPPING_ALL.get(lbl)


# ------------------------------------------------------------
# 6. Load benchmark JSONL
# ------------------------------------------------------------

def load_all_benchmark(file_path):

    records = []

    with open(
        file_path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            row = json.loads(line)

            mapped_label = map_all_label(
                row
            )

            # Ignore languages outside the selected benchmark set.
            if mapped_label is not None:

                row["target_label"] = mapped_label

                records.append(
                    row
                )

    return pd.DataFrame(
        records
    )


# ------------------------------------------------------------
# 7. Predict using fine-tuned ConLID
# ------------------------------------------------------------

import gc
import torch

MAX_TEXT_CHARS = 2000

def predict_texts(texts, batch_size=8):

    predictions = []

    # Clear unused GPU memory before evaluation
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    model.eval()

    for start in tqdm(
        range(0, len(texts), batch_size),
        desc="Predicting"
    ):

        # Same text-length limit used during training
        chunk = [
            str(text)[:MAX_TEXT_CHARS]
            for text in texts[start:start + batch_size]
        ]

        with torch.inference_mode():

            labels_batch, _ = model.predict_batched(
                chunk,
                k=1
            )

        for labels in labels_batch:

            if labels:
                model_label = labels[0]
            else:
                model_label = "unknown"

            readable_label = MODEL_LABEL_TO_NAME.get(
                model_label,
                model_label
            )

            predictions.append(
                readable_label
            )

        # Periodically release unused cached memory
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return predictions




# ------------------------------------------------------------
# 8. Evaluate each benchmark
# ------------------------------------------------------------

import gc
import torch

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print(
    "\nEvaluating fine-tuned ConLID..."
)

model.eval()

all_language_summary = []


for file_path in benchmark_files:

    dataset_name = file_path.stem

    df_all = load_all_benchmark(
        file_path
    )

    if df_all.empty:

        print(
            f"\nNo matching rows found in "
            f"{dataset_name}."
        )

        continue


    print("\n" + "=" * 70)
    print(
        f"DATASET: {dataset_name}"
    )
    print("=" * 70)

    print(
        f"Rows selected: {len(df_all)}"
    )

    print(
        "\nTrue-label distribution:"
    )

    print(
        df_all["target_label"]
        .value_counts()
    )


    # ----------------------------
    # Predictions
    # ----------------------------

    results = df_all.copy()

    results["predicted_label"] = (
        predict_texts(
            results["text"].tolist()
        )
    )


    y_true = (
        results["target_label"]
        .tolist()
    )

    y_pred = (
        results["predicted_label"]
        .tolist()
    )


    # ----------------------------
    # Overall metrics
    # ----------------------------

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    present_languages = [
        language for language in ALL_BENCHMARK_LANGUAGES
        if language in set(y_true)
    ]


    macro_f1 = f1_score(
        y_true,
        y_pred,
        labels=present_languages,
        average="macro",
        zero_division=0
    )


    print("\n" + "-" * 70)

    print(
        f"Accuracy: "
        f"{accuracy:.4f} "
        f"({accuracy * 100:.2f}%)"
    )

    print(
        f"Macro F1: "
        f"{macro_f1:.4f} "
        f"({macro_f1 * 100:.2f}%)"
    )

    print("-" * 70)


    # ----------------------------
    # Per-language metrics
    # ----------------------------

    print(
        "\nPer-language results:\n"
    )

    print(
        classification_report(
            y_true,
            y_pred,
            labels=present_languages,
            digits=4,
            zero_division=0
        )
    )


    # ----------------------------
    # Save predictions
    # ----------------------------

    output_csv = (
        BENCHMARK_OUTPUT_DIR
        / f"conlid_finetuned_all_langs_{dataset_name}.csv"
    )

    results.to_csv(
        output_csv,
        index=False
    )

    print(
        f"\nSaved predictions to:\n"
        f"{output_csv}"
    )


    # ----------------------------
    # Summary
    # ----------------------------

    all_language_summary.append(
        {
            "dataset": dataset_name,
            "rows": len(results),
            "accuracy": accuracy,
            "macro_f1": macro_f1,
        }
    )


# ------------------------------------------------------------
# 9. Final benchmark summary
# ------------------------------------------------------------

all_language_summary_df = pd.DataFrame(
    all_language_summary
)

print("\n" + "=" * 70)
print("FINAL BENCHMARK SUMMARY")
print("=" * 70)

display(
    all_language_summary_df
)


# Save summary
summary_path = (
    BENCHMARK_OUTPUT_DIR
    / "conlid_finetuned_benchmark_summary.csv"
)

all_language_summary_df.to_csv(
    summary_path,
    index=False
)

print(
    f"\nSummary saved to:\n"
    f"{summary_path}"
)

Benchmark directory:
/content/drive/MyDrive/SSLI/processed/public_shared/benchmark_datasets

Benchmark files:
 - commonlid.jsonl
 - flores_plus.jsonl
 - wili-2018.jsonl

Results will be saved to:
/content/drive/MyDrive/SSLI/benchmark_results/conlid_finetuned_script_fixed
GPU memory allocated: 2.16 GB

Evaluating fine-tuned ConLID...

DATASET: commonlid
Rows selected: 77974

True-label distribution:
target_label
english          27461
arabic           26152
german            7553
hindi             3666
french            3233
pali              3027
sinhala           2693
bengali           1886
sanskrit          1327
sanskrit_deva      895
tamil               81
Name: count, dtype: int64


Predicting:   0%|          | 0/9747 [00:00<?, ?it/s]


----------------------------------------------------------------------
Accuracy: 0.7570 (75.70%)
Macro F1: 0.9232 (92.32%)
----------------------------------------------------------------------

Per-language results:

               precision    recall  f1-score   support

      sinhala     0.9602    0.9684    0.9643      2693
         pali     0.9735    0.9706    0.9720      3027
     sanskrit     0.9767    0.9179    0.9464      1327
sanskrit_deva     0.9791    0.9430    0.9607       895
      english     0.9963    0.7381    0.8480     27461
        tamil     0.9643    1.0000    0.9818        81
        hindi     0.9957    0.8914    0.9407      3666
      bengali     1.0000    0.9215    0.9592      1886
       arabic     0.9998    0.6627    0.7971     26152
       french     0.9827    0.8252    0.8971      3233
       german     0.9933    0.8034    0.8883      7553

    micro avg     0.9930    0.7570    0.8591     77974
    macro avg     0.9838    0.8766    0.9232     77974
 weighted

Predicting:   0%|          | 0/2020 [00:00<?, ?it/s]


----------------------------------------------------------------------
Accuracy: 0.8872 (88.72%)
Macro F1: 0.9336 (93.36%)
----------------------------------------------------------------------

Per-language results:

               precision    recall  f1-score   support

      sinhala     0.9602    0.9684    0.9643      2693
         pali     0.9735    0.9706    0.9720      3027
     sanskrit     0.9767    0.9179    0.9464      1327
sanskrit_deva     1.0000    0.9970    0.9985      1012
      english     1.0000    0.9921    0.9960      1012
        tamil     0.9990    1.0000    0.9995      1012
        hindi     0.9970    0.9970    0.9970      1012
      bengali     1.0000    0.9990    0.9995      1012
       arabic     0.9961    0.2500    0.3997      2024
       french     1.0000    1.0000    1.0000      1012
       german     1.0000    0.9941    0.9970      1012

    micro avg     0.9847    0.8872    0.9334     16155
    macro avg     0.9911    0.9169    0.9336     16155
 weighted

Predicting:   0%|          | 0/1756 [00:00<?, ?it/s]


----------------------------------------------------------------------
Accuracy: 0.9611 (96.11%)
Macro F1: 0.9695 (96.95%)
----------------------------------------------------------------------

Per-language results:

               precision    recall  f1-score   support

      sinhala     0.9602    0.9684    0.9643      2693
         pali     0.9735    0.9706    0.9720      3027
     sanskrit     0.9767    0.9179    0.9464      1327
sanskrit_deva     1.0000    0.9850    0.9924      1000
      english     0.9146    0.9740    0.9433      1000
        tamil     0.9990    0.9900    0.9945      1000
        hindi     1.0000    0.9800    0.9899      1000
      bengali     1.0000    0.8930    0.9435      1000
       french     0.9859    0.9760    0.9809      1000
       german     0.9989    0.9390    0.9680      1000

    micro avg     0.9766    0.9611    0.9688     14047
    macro avg     0.9809    0.9594    0.9695     14047
 weighted avg     0.9772    0.9611    0.9687     14047


Saved p

,dataset,rows,accuracy,macro_f1
0,commonlid,77974,0.757047,0.923238
1,flores_plus,16155,0.887218,0.933641
2,wili-2018,14047,0.961130,0.969533



Summary saved to:
/content/drive/MyDrive/SSLI/benchmark_results/conlid_finetuned_script_fixed/conlid_finetuned_benchmark_summary.csv


In [ ]:
print("san_Sinh ->", MODEL_LABEL_TO_NAME.get("san_Sinh"))
print("san_Deva ->", MODEL_LABEL_TO_NAME.get("san_Deva"))

print("\nLabels inside model:")
print("san_Sinh exists:", "san_Sinh" in model.id2label.values())
print("san_Deva exists:", "san_Deva" in model.id2label.values())

san_Sinh -> sanskrit
san_Deva -> sanskrit_deva

Labels inside model:
san_Sinh exists: True
san_Deva exists: True


In [ ]:
import json
import pandas as pd

FLORES_PATH = (
    "/content/drive/MyDrive/SSLI/processed/public_shared/"
    "benchmark_datasets/flores_plus.jsonl"
)

rows = []

with open(FLORES_PATH, encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)

        if str(row.get("label", "")).strip() == "san":
            rows.append(row)

san_df = pd.DataFrame(rows)

print("Total Sanskrit rows:", len(san_df))
print("\nSources:")
print(san_df["source"].value_counts())

In [ ]:
def detect_script(text):
    sinhala = 0
    devanagari = 0

    for ch in str(text):
        code = ord(ch)

        if 0x0D80 <= code <= 0x0DFF:
            sinhala += 1

        elif 0x0900 <= code <= 0x097F:
            devanagari += 1

    if sinhala > devanagari:
        return "Sinhala"

    if devanagari > sinhala:
        return "Devanagari"

    return "Other"


san_df["detected_script"] = (
    san_df["text"].apply(detect_script)
)

print(
    pd.crosstab(
        san_df["source"],
        san_df["detected_script"]
    )
)

In [ ]:
def raw_predict(texts, batch_size=8):
    outputs = []

    for start in range(0, len(texts), batch_size):

        chunk = [
            str(x)[:2000]
            for x in texts[start:start + batch_size]
        ]

        labels_batch, _ = model.predict_batched(
            chunk,
            k=1
        )

        for labels in labels_batch:
            outputs.append(
                labels[0] if labels else "unknown"
            )

    return outputs


sample = san_df.groupby(
    "detected_script",
    group_keys=False
).head(10).copy()

sample["raw_prediction"] = raw_predict(
    sample["text"].tolist()
)

sample[
    [
        "source",
        "detected_script",
        "raw_prediction",
        "text"
    ]
]

In [ ]:
RESULT_PATH = (
    "/content/drive/MyDrive/SSLI/"
    "benchmark_results/conlid_finetuned_script_fixed/"
    "conlid_finetuned_all_langs_flores_plus.csv"
)

results = pd.read_csv(RESULT_PATH)

sanskrit_results = results[
    results["target_label"].isin(
        ["sanskrit", "sanskrit_deva"]
    )
]

print(
    pd.crosstab(
        sanskrit_results["target_label"],
        sanskrit_results["predicted_label"],
        margins=True
    )
)

## Download the trained model

The cell below zips the saved model so it can be downloaded from Colab in one
piece and then uploaded to the Hugging Face Hub from your local machine.


In [ ]:
# ============================================================
# PACKAGE THE MODEL FOR DOWNLOAD / UPLOAD
# ============================================================
import shutil
from pathlib import Path

out_dir = Path(output_model_dir)

# Confirm the model really is on disk before packaging it.
verify_saved_model(out_dir)

archive_base = str(out_dir.parent / (out_dir.name + "_bundle"))
archive_path = shutil.make_archive(archive_base, "zip", root_dir=str(out_dir))

size_mb = os.path.getsize(archive_path) / (1024 * 1024)
print(f"\nArchive: {archive_path}")
print(f"Size:    {size_mb:,.1f} MB")

if size_mb < 100:
    raise RuntimeError(
        f"Archive is only {size_mb:.1f} MB - far smaller than the expected "
        "~1.1 GB. The model weights are missing; do not download this."
    )

print("\nDownload this archive from the Colab file browser (or Drive),")
print("extract it locally, then upload with:")
print()
print("  hf repo create script-langid/conlid-2101-sinhala-pali-sanskrit --repo-type model")
print('  hf upload script-langid/conlid-2101-sinhala-pali-sanskrit "<extracted folder>" . --repo-type model')
